In [19]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph,END, START
from typing import TypedDict

In [20]:
OLLAMA_MODEL = "qwen2.5-coder:14b"
OLLAMA_BASE_URL = "http://localhost:11434"

In [21]:
model = ChatOllama(
    model=OLLAMA_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0,
)

In [22]:
class ConvType(TypedDict):
    topic:str
    outline:str
    blog:str

In [23]:
def outline_generation(state: ConvType):
    topic = state["topic"]

    template = PromptTemplate(
        input_variables=["topic"],
        template="You are a helpful AI that gives short but engaging GenZ-style answers. Use slang to give a one liner and engaging outline for this topic. Topic: {topic}"
    )

    prompt = template.format(topic=topic)
    response=model.invoke(prompt).content

    return {"outline": response}

In [24]:
def blog_generation(state: ConvType):
    topic = state["topic"]
    outline=state['outline']

    template = PromptTemplate(
        input_variables=["topic","outline"],
        template="You are a helpful AI that gives short but engaging GenZ-style answers. Use slang to give a comprehensive blog about 500 words based on the {topic} and the outline {outline}"
    )

    prompt = template.format(topic=topic, outline=outline)
    response=model.invoke(prompt).content

    return {"blog": response}

In [25]:
graph= StateGraph(ConvType)

graph.add_node('outline_generation',outline_generation)
graph.add_node('blog_generation',blog_generation)
graph.add_edge(START,"outline_generation")
graph.add_edge("outline_generation","blog_generation")
graph.add_edge("blog_generation",END)

In [26]:
workflow=graph.compile()
final_node=workflow.invoke({'topic':"blackholes"})

In [27]:
print(final_node["outline"])

Black holes rly lit 🔥 - they're these cosmic vacuums where gravity is so strong, not even light can escape. Imagine a super duper dense ball that warps space-time around it. They come in different sizes too, from tiny ones called "stellar" to massive galactic behemoths. Scientists are still figuring out all the mysteries of black holes, but they're basically nature's ultimate trash cans for matter and energy!


In [28]:
print(final_node["blog"])

Hey y'all! 🔥 So, let’s dive into the wild world of black holes. These cosmic vacuums are like the ultimate party poopers in space—gravity so strong that not even light can escape. Imagine a super duper dense ball that warps space-time around it. It's like if you had a bowling ball on a trampoline, and everything around it got sucked down into a deep pit. That’s kinda what black holes do to the universe.

There are different sizes of black holes too. You’ve got your tiny ones called "stellar" black holes, which form when big stars die and collapse in on themselves. They’re like the size of a city. Then you've got these massive galactic behemoths that can be millions or even billions of times bigger than our sun. These are the real deal—think of them as cosmic monsters lurking in the dark corners of galaxies.

Scientists are still figuring out all the mysteries of black holes, but they’re basically nature’s ultimate trash cans for matter and energy! Everything that gets too close gets su